In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.model_selection import RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings


In [3]:
df = pd.read_csv("StudentsPerformance.csv")

In [4]:
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


## predict math score

In [5]:
x = df.drop(columns=["math score"],axis =.1)

In [6]:
y = df["math score"]

In [7]:
x

,gender,race/ethnicity,parental level of education,lunch,test preparation course,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75
...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,99,95
996,male,group C,high school,free/reduced,none,55,55
997,female,group C,high school,free/reduced,completed,71,65
998,female,group D,some college,standard,completed,78,77


In [8]:
y

0      72
1      69
2      90
3      47
4      76
       ..
995    88
996    62
997    59
998    68
999    77
Name: math score, Length: 1000, dtype: int64

In [14]:
num_feature = x.select_dtypes(exclude="object").columns.tolist()

In [18]:
cat_feature = x.select_dtypes(include="object").columns.tolist()

In [16]:
num_feature

['reading score', 'writing score']

In [19]:
cat_feature

['gender',
 'race/ethnicity',
 'parental level of education',
 'lunch',
 'test preparation course']

In [20]:
preprocessor = ColumnTransformer(
    transformers=[("OneHotEncoder",OneHotEncoder(drop = "first",sparse_output=False),cat_feature)
                  ,("StandardScaler",StandardScaler(),num_feature)]
,remainder = "passthrough")

In [23]:
x_encoded = preprocessor.fit_transform(x)


In [24]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    x_encoded, y, test_size=0.2, random_state=42
)

In [25]:
print(y.head())
print(y.dtype)
print(y.nunique())

0    72
1    69
2    90
3    47
4    76
Name: math score, dtype: int64
int64
81


In [26]:
lr = LinearRegression()
lr.fit(X_train,y_train)

LinearRegression()

In [27]:
y_pred = lr.predict(X_test)

/Users/dhruv/Downloads/python_practice/Project_1/.myenv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/dhruv/Downloads/python_practice/Project_1/.myenv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/dhruv/Downloads/python_practice/Project_1/.myenv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [28]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"R² Score: {r2:.4f}")
print(f"MAE:      {mae:.2f} marks")
print(f"RMSE:     {rmse:.2f} marks")

R² Score: 0.8804
MAE:      4.21 marks
RMSE:     5.39 marks


In [29]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)

lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)

print(f"Ridge R²: {r2_score(y_test, y_pred_ridge):.4f}")
print(f"Lasso R²: {r2_score(y_test, y_pred_lasso):.4f}")

Ridge R²: 0.8805
Lasso R²: 0.8816


/Users/dhruv/Downloads/python_practice/Project_1/.myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/dhruv/Downloads/python_practice/Project_1/.myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/dhruv/Downloads/python_practice/Project_1/.myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/dhruv/Downloads/python_practice/Project_1/.myenv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/dhruv/Downloads/python_practice/Project_1/.myenv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/dhruv/Downloads/python_practice/Project_1/.my

In [30]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor(n_neighbors=7)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)
print(f"KNN R²: {r2_score(y_test, y_pred_knn):.4f}")

KNN R²: 0.7756


In [31]:
dt = DecisionTreeRegressor(random_state=42)

# 2. Train data par fit karein
dt.fit(X_train, y_train)

# 3. Train aur Test dono par predict karein
y_train_pred = dt.predict(X_train)
y_test_pred = dt.predict(X_test)

# 4. Metrics print karein
print(f"Train R² Score: {r2_score(y_train, y_train_pred):.4f}")
print(f"Test R² Score:  {r2_score(y_test, y_test_pred):.4f}")
print(f"Test MAE:       {mean_absolute_error(y_test, y_test_pred):.2f} marks")
print(f"Test RMSE:      {np.sqrt(mean_squared_error(y_test, y_test_pred)):.2f} marks")


Train R² Score: 0.9997
Test R² Score:  0.7313
Test MAE:       6.41 marks
Test RMSE:      8.09 marks


In [32]:
svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)

# 2. Fit karein
svr.fit(X_train, y_train)

# 3. Predict karein
y_train_pred_svr = svr.predict(X_train)
y_test_pred_svr = svr.predict(X_test)

# 4. Metrics print karein
print(f"Train R² Score: {r2_score(y_train, y_train_pred_svr):.4f}")
print(f"Test R² Score:  {r2_score(y_test, y_test_pred_svr):.4f}")
print(f"Test MAE:       {mean_absolute_error(y_test, y_test_pred_svr):.2f} marks")
print(f"Test RMSE:      {np.sqrt(mean_squared_error(y_test, y_test_pred_svr)):.2f} marks")

Train R² Score: 0.7982
Test R² Score:  0.7130
Test MAE:       5.50 marks
Test RMSE:      8.36 marks


In [33]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)

# 2. Train data par fit karein
rf.fit(X_train, y_train)

# 3. Train aur Test dono par predict karein
y_train_pred_rf = rf.predict(X_train)
y_test_pred_rf = rf.predict(X_test)

# 4. Metrics print karein
print(f"Train R² Score: {r2_score(y_train, y_train_pred_rf):.4f}")
print(f"Test R² Score:  {r2_score(y_test, y_test_pred_rf):.4f}")
print(f"Test MAE:       {mean_absolute_error(y_test, y_test_pred_rf):.2f} marks")
print(f"Test RMSE:      {np.sqrt(mean_squared_error(y_test, y_test_pred_rf)):.2f} marks")

Train R² Score: 0.9761
Test R² Score:  0.8488
Test MAE:       4.75 marks
Test RMSE:      6.07 marks


In [35]:

# 1. Initialize Gradient Boosting
gb = AdaBoostRegressor(random_state=42)

# 2. Fit on training data
gb.fit(X_train, y_train)

# 3. Predict on train and test
y_train_pred_gb = gb.predict(X_train)
y_test_pred_gb = gb.predict(X_test)

# 4. Metrics
print(f"Train R² Score: {r2_score(y_train, y_train_pred_gb):.4f}")
print(f"Test R² Score:  {r2_score(y_test, y_test_pred_gb):.4f}")
print(f"Test MAE:       {mean_absolute_error(y_test, y_test_pred_gb):.2f} marks")
print(f"Test RMSE:      {np.sqrt(mean_squared_error(y_test, y_test_pred_gb)):.2f} marks")

Train R² Score: 0.8491
Test R² Score:  0.8507
Test MAE:       4.64 marks
Test RMSE:      6.03 marks


In [36]:
from sklearn.ensemble import GradientBoostingRegressor

# 1. Gradient Boosting initialize karein
gb = GradientBoostingRegressor(random_state=42)

# 2. Fit on training data
gb.fit(X_train, y_train)

# 3. Predict on train and test
y_train_pred_gb = gb.predict(X_train)
y_test_pred_gb = gb.predict(X_test)

# 4. Metrics print karein
print(f"Train R² Score: {r2_score(y_train, y_train_pred_gb):.4f}")
print(f"Test R² Score:  {r2_score(y_test, y_test_pred_gb):.4f}")
print(f"Test MAE:       {mean_absolute_error(y_test, y_test_pred_gb):.2f} marks")
print(f"Test RMSE:      {np.sqrt(mean_squared_error(y_test, y_test_pred_gb)):.2f} marks")

Train R² Score: 0.9030
Test R² Score:  0.8724
Test MAE:       4.30 marks
Test RMSE:      5.57 marks


In [37]:
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

# Fit on preprocessed training data
xgb_model.fit(X_train, y_train)

# Predict
y_train_xgb = xgb_model.predict(X_train)
y_test_xgb = xgb_model.predict(X_test)

print(f"XGBoost Train R²: {r2_score(y_train, y_train_xgb):.4f}")
print(f"XGBoost Test R²:  {r2_score(y_test, y_test_xgb):.4f}")
print(f"XGBoost Test MAE: {mean_absolute_error(y_test, y_test_xgb):.2f}")
print(f"XGBoost Test RMSE:{np.sqrt(mean_squared_error(y_test, y_test_xgb)):.2f}")


XGBoost Train R²: 0.9028
XGBoost Test R²:  0.8677
XGBoost Test MAE: 4.31
XGBoost Test RMSE:5.67


In [38]:
cb_model = CatBoostRegressor(
    iterations=200,
    learning_rate=0.05,
    depth=4,
    verbose=0,
    random_state=42
)

# Fit on the preprocessed training data
cb_model.fit(X_train, y_train)

# Predict
y_train_cb = cb_model.predict(X_train)
y_test_cb = cb_model.predict(X_test)

print(f"CatBoost Train R²: {r2_score(y_train, y_train_cb):.4f}")
print(f"CatBoost Test R²:  {r2_score(y_test, y_test_cb):.4f}")
print(f"CatBoost Test MAE: {mean_absolute_error(y_test, y_test_cb):.2f}")
print(f"CatBoost Test RMSE:{np.sqrt(mean_squared_error(y_test, y_test_cb)):.2f}")

CatBoost Train R²: 0.8880
CatBoost Test R²:  0.8683
CatBoost Test MAE: 4.36
CatBoost Test RMSE:5.66
